Clean the data, prepare and create the final table

In [0]:
from pyspark.sql import functions as F
file_path = "/Volumes/main/christma_hits_schema/christmas_hits_schema"
df_raw = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv(file_path)
)

df_raw.printSchema()
df_raw.show(5)


We are going to turn the week_of_year and streams columns into float to make calculations easier

In [0]:
df = (
    df_raw
      .withColumn("streams", F.col("streams").cast("double"))
      .withColumn("week_of_year", F.col("week_of_year").cast("double"))
      
)
df.printSchema()
df.show(5)



Now we create the managed table

In [0]:
df.write \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("main.christma_hits_schema.christmas_hits")



SQL Analysis

In [0]:
%sql 
SELECT  week_of_year,
        SUM(streams) AS weekly_streams
FROM main.christma_hits_schema.christmas_hits
GROUP BY week_of_year
ORDER BY weekly_streams DESC;


In [0]:
%sql 
SELECT COUNT(DISTINCT track) AS number_of_hits
FROM main.christma_hits_schema.christmas_hits


In [0]:
%sql
SELECT 
    artist,
    COUNT(*) AS artist_count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM main.christma_hits_schema.christmas_hits
GROUP BY artist
ORDER BY percentage DESC;

In [0]:
%sql 
SELECT  date,
        COUNT(date) AS christmas_mood_days
FROM main.christma_hits_schema.christmas_hits
GROUP BY date
ORDER BY christmas_mood_days DESC;

Machine Learning model using regression

In [0]:
#libraries
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

#data
df = spark.table("main.christma_hits_schema.christmas_hits")

#we scale the streams so that we dont get a high RMSE
df = df.withColumn("streams_millions", F.col("streams") / 1_000_000)


df = df.withColumn("week_of_year_squared", F.col("week_of_year")**2)

# features vector
assembler = VectorAssembler(
    inputCols=["week_of_year", "week_of_year_squared"],
    outputCol="features"
)
df = assembler.transform(df).select("features", "streams_millions")

#train and test
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

#model training
lr = LinearRegression(featuresCol="features", labelCol="streams_millions")
lr_model = lr.fit(train_df)

# predictions
predictions = lr_model.transform(test_df)
predictions.select("features", "streams_millions", "prediction").show(5)

# evaluation
rmse = RegressionEvaluator(
    labelCol="streams_millions",
    predictionCol="prediction",
    metricName="rmse"
).evaluate(predictions)

r2 = RegressionEvaluator(
    labelCol="streams_millions",
    predictionCol="prediction",
    metricName="r2"
).evaluate(predictions)

print(f"RMSE (millones): {rmse}")
print(f"R²: {r2}")

# visualization
display(predictions.select("features", "streams_millions", "prediction"))

